# Mean Reversion Strategy - Statistical Arbitrage Deep Dive

This notebook explores **mean reversion strategies** - a fundamental approach in statistical arbitrage that profits from prices returning to their historical average.

## Philosophy: The Opposite of Momentum

Mean reversion is the **inverse** of momentum:
- **Momentum**: "What goes up keeps going up" → buy winners, sell losers
- **Mean Reversion**: "What goes up must come down" → sell winners, buy losers

## What You'll Learn

1. **Z-Score Based Entry/Exit Signals** - Statistical thresholds for trading
2. **Bollinger Bands** - Volatility-adjusted mean reversion bands
3. **Half-Life Estimation** - Measuring mean reversion speed (Ornstein-Uhlenbeck process)
4. **Entry Threshold Optimization** - Testing |z| > 1.5, 2.0, 2.5 thresholds
5. **Sector-Neutral Positioning** - Eliminating systematic risk
6. **High Turnover Management** - Controlling transaction costs
7. **Stop-Loss Implementation** - Protecting against trending breakouts
8. **When Mean Reversion Works vs Fails** - Regime detection

## Key Insight

Mean reversion works in **ranging markets** but fails catastrophically in **trending markets**. Knowing when to use it is as important as knowing how.

---

## Setup

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Core libraries
import numpy as np
import pandas as pd
import polars as pl
from datetime import date, timedelta

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ARBS framework
from Signals.Futures.MeanReversionSignal import MeanReversionSignal
from Signals.Futures.MomentumSignal import MomentumSignal
from Signals.Utils.IC import calculate_ic
from Strategies.Registry import quick_strategy

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Random seed for reproducibility
np.random.seed(42)

print("✓ Setup complete!")

---

## 1. Understanding Mean Reversion: The Ornstein-Uhlenbeck Process

The **Ornstein-Uhlenbeck (OU) process** is the mathematical foundation for mean reversion:

$$dX_t = \theta(\mu - X_t)dt + \sigma dW_t$$

Where:
- $X_t$ = price (or spread) at time t
- $\theta$ = mean reversion speed (higher = faster reversion)
- $\mu$ = long-term mean
- $\sigma$ = volatility
- $W_t$ = Wiener process (random walk)

### Half-Life Formula

The **half-life** tells us how long it takes for a deviation to shrink by 50%:

$$\tau_{1/2} = \frac{-\ln(2)}{\theta}$$

**Interpretation**:
- Short half-life (1-5 days): High-frequency mean reversion → intraday/daily strategies
- Medium half-life (5-20 days): Swing trading opportunities
- Long half-life (>30 days): Slow mean reversion → not ideal for trading

In [ ]:
def simulate_ou_process(theta, mu, sigma, T, dt, X0):
    """
    Simulate Ornstein-Uhlenbeck process.
    
    Args:
        theta: Mean reversion speed
        mu: Long-term mean
        sigma: Volatility
        T: Total time
        dt: Time step
        X0: Initial value
    
    Returns:
        Array of simulated prices
    """
    n_steps = int(T / dt)
    X = np.zeros(n_steps)
    X[0] = X0
    
    for i in range(1, n_steps):
        dW = np.random.normal(0, np.sqrt(dt))
        dX = theta * (mu - X[i-1]) * dt + sigma * dW
        X[i] = X[i-1] + dX
    
    return X

def estimate_half_life(prices):
    """
    Estimate half-life from price series using AR(1) model.
    
    AR(1): X_t = ρ*X_{t-1} + ε
    Half-life = -ln(2) / ln(ρ)
    """
    # Fit AR(1) model: regress prices on lagged prices
    X_lag = prices[:-1]
    X_current = prices[1:]
    
    # Demean the series
    X_lag_dm = X_lag - X_lag.mean()
    X_current_dm = X_current - X_current.mean()
    
    # Calculate autocorrelation coefficient (ρ)
    rho = np.sum(X_lag_dm * X_current_dm) / np.sum(X_lag_dm ** 2)
    
    # Ensure rho is in valid range for mean reversion (0 < rho < 1)
    if rho <= 0 or rho >= 1:
        return np.inf  # Not mean-reverting
    
    # Calculate half-life
    half_life = -np.log(2) / np.log(rho)
    
    return half_life

# Simulate three OU processes with different reversion speeds
T = 252  # 1 year
dt = 1   # Daily
mu = 100  # Mean price
sigma = 2  # Volatility
X0 = 110  # Start above mean

# Fast, medium, slow mean reversion
theta_fast = 0.3    # Half-life ≈ 2.3 days
theta_medium = 0.1  # Half-life ≈ 6.9 days
theta_slow = 0.03   # Half-life ≈ 23.1 days

X_fast = simulate_ou_process(theta_fast, mu, sigma, T, dt, X0)
X_medium = simulate_ou_process(theta_medium, mu, sigma, T, dt, X0)
X_slow = simulate_ou_process(theta_slow, mu, sigma, T, dt, X0)

# Estimate half-lives from simulated data
hl_fast = estimate_half_life(X_fast)
hl_medium = estimate_half_life(X_medium)
hl_slow = estimate_half_life(X_slow)

# Plot the three processes
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(X_fast, linewidth=1.5, color='steelblue', label=f'Fast (θ={theta_fast})')
axes[0].axhline(mu, color='red', linestyle='--', alpha=0.5, label=f'Mean = {mu}')
axes[0].set_title(f'Fast Mean Reversion (Estimated Half-Life: {hl_fast:.1f} days)', fontweight='bold')
axes[0].set_ylabel('Price')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(X_medium, linewidth=1.5, color='green', label=f'Medium (θ={theta_medium})')
axes[1].axhline(mu, color='red', linestyle='--', alpha=0.5, label=f'Mean = {mu}')
axes[1].set_title(f'Medium Mean Reversion (Estimated Half-Life: {hl_medium:.1f} days)', fontweight='bold')
axes[1].set_ylabel('Price')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(X_slow, linewidth=1.5, color='orange', label=f'Slow (θ={theta_slow})')
axes[2].axhline(mu, color='red', linestyle='--', alpha=0.5, label=f'Mean = {mu}')
axes[2].set_title(f'Slow Mean Reversion (Estimated Half-Life: {hl_slow:.1f} days)', fontweight='bold')
axes[2].set_ylabel('Price')
axes[2].set_xlabel('Time (days)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Half-Life Analysis:")
print("=" * 60)
print(f"Fast reversion (θ={theta_fast}):     Theoretical = {-np.log(2)/np.log(1-theta_fast*dt):.1f} days, Estimated = {hl_fast:.1f} days")
print(f"Medium reversion (θ={theta_medium}):  Theoretical = {-np.log(2)/np.log(1-theta_medium*dt):.1f} days, Estimated = {hl_medium:.1f} days")
print(f"Slow reversion (θ={theta_slow}):    Theoretical = {-np.log(2)/np.log(1-theta_slow*dt):.1f} days, Estimated = {hl_slow:.1f} days")
print("\n💡 Key Insight: Faster mean reversion = quicker return to mean = more trading opportunities")

---

## 2. Z-Score Based Entry/Exit Signals

The **Z-score** measures how many standard deviations a price is from its mean:

$$Z_t = \frac{P_t - \bar{P}}{\sigma_P}$$

**Trading Rules**:
- **Buy (Long)**: Z < -2 (price is 2 std devs below mean → oversold)
- **Sell (Short)**: Z > +2 (price is 2 std devs above mean → overbought)
- **Exit**: Z crosses zero (price returns to mean)

**Question**: Should we use |Z| > 1.5, 2.0, or 2.5 as entry threshold?
- Lower threshold (1.5): More trades, higher turnover, lower signal quality
- Higher threshold (2.5): Fewer trades, lower turnover, higher signal quality

Let's test this empirically!

In [ ]:
def calculate_zscore(prices, window=20):
    """
    Calculate rolling Z-score.
    
    Args:
        prices: Price series
        window: Lookback window for mean/std calculation
    
    Returns:
        Z-score series
    """
    rolling_mean = pd.Series(prices).rolling(window=window).mean()
    rolling_std = pd.Series(prices).rolling(window=window).std()
    
    zscore = (prices - rolling_mean) / rolling_std
    
    return zscore

def generate_mean_reversion_signals(zscore, threshold=2.0):
    """
    Generate entry/exit signals based on Z-score threshold.
    
    Args:
        zscore: Z-score series
        threshold: Entry threshold (e.g., 2.0 for ±2 sigma)
    
    Returns:
        Signal series: +1 (long), -1 (short), 0 (neutral)
    """
    signals = np.zeros_like(zscore)
    
    # Buy when Z < -threshold (oversold)
    signals[zscore < -threshold] = 1
    
    # Sell when Z > +threshold (overbought)
    signals[zscore > threshold] = -1
    
    return signals

# Simulate mean-reverting price with occasional trends
n_periods = 500
prices = np.zeros(n_periods)
prices[0] = 100

# Mean-reverting component with occasional trend breaks
theta = 0.1
mu = 100
sigma = 2

for i in range(1, n_periods):
    # Add a trend break every 100 periods
    if i % 100 == 0:
        mu += np.random.choice([-5, 5])  # Shift the mean
    
    dW = np.random.normal(0, 1)
    dX = theta * (mu - prices[i-1]) + sigma * dW
    prices[i] = prices[i-1] + dX

# Calculate Z-scores with different lookback windows
window = 20
zscore = calculate_zscore(prices, window=window)

# Test different thresholds
thresholds = [1.5, 2.0, 2.5]
signals_dict = {}

for threshold in thresholds:
    signals = generate_mean_reversion_signals(zscore, threshold=threshold)
    signals_dict[threshold] = signals

# Visualization
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)

# 1. Price series
axes[0].plot(prices, linewidth=1.5, color='black', label='Price')
axes[0].set_title('Simulated Mean-Reverting Price (with regime shifts)', fontweight='bold')
axes[0].set_ylabel('Price')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Z-score
axes[1].plot(zscore, linewidth=1.5, color='steelblue', label='Z-score')
axes[1].axhline(0, color='black', linestyle='-', alpha=0.3)
axes[1].axhline(2, color='red', linestyle='--', alpha=0.5, label='±2σ')
axes[1].axhline(-2, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(1.5, color='orange', linestyle='--', alpha=0.3, label='±1.5σ')
axes[1].axhline(-1.5, color='orange', linestyle='--', alpha=0.3)
axes[1].axhline(2.5, color='purple', linestyle='--', alpha=0.3, label='±2.5σ')
axes[1].axhline(-2.5, color='purple', linestyle='--', alpha=0.3)
axes[1].set_title('Z-Score (20-day rolling)', fontweight='bold')
axes[1].set_ylabel('Z-score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3-5. Signals for different thresholds
for idx, threshold in enumerate(thresholds):
    signals = signals_dict[threshold]
    
    # Color code: green = buy, red = sell, gray = neutral
    colors = ['green' if s > 0 else 'red' if s < 0 else 'lightgray' for s in signals]
    
    axes[2+idx].scatter(range(len(signals)), signals, c=colors, alpha=0.3, s=10)
    axes[2+idx].set_title(f'Signals (threshold = {threshold}σ)', fontweight='bold')
    axes[2+idx].set_ylabel('Signal')
    axes[2+idx].set_ylim(-1.5, 1.5)
    axes[2+idx].axhline(0, color='black', linestyle='-', alpha=0.3)
    axes[2+idx].grid(True, alpha=0.3)
    
    # Count signals
    n_long = np.sum(signals > 0)
    n_short = np.sum(signals < 0)
    axes[2+idx].text(0.02, 0.95, f'Long: {n_long}, Short: {n_short}, Total: {n_long + n_short}',
                     transform=axes[2+idx].transAxes, verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[-1].set_xlabel('Time (periods)')

plt.tight_layout()
plt.show()

print("\n📊 Signal Statistics by Threshold:")
print("=" * 60)
for threshold in thresholds:
    signals = signals_dict[threshold]
    n_total = np.sum(signals != 0)
    pct_active = n_total / len(signals) * 100
    print(f"Threshold {threshold}σ: {n_total} signals ({pct_active:.1f}% of time in positions)")

print("\n💡 Key Insight: Lower threshold = more trades but lower signal quality")
print("   Higher threshold = fewer trades but stronger mean reversion signals")

---

## 3. Bollinger Bands Implementation

**Bollinger Bands** are volatility-adjusted mean reversion bands:

- **Middle Band**: 20-day moving average
- **Upper Band**: Middle + 2 × standard deviation
- **Lower Band**: Middle - 2 × standard deviation

**Trading Rules**:
- Buy when price touches lower band (oversold)
- Sell when price touches upper band (overbought)
- Exit when price crosses middle band

**Advantage over fixed Z-score**: Bands adapt to changing volatility!

In [ ]:
def calculate_bollinger_bands(prices, window=20, num_std=2):
    """
    Calculate Bollinger Bands.
    
    Args:
        prices: Price series
        window: Moving average window
        num_std: Number of standard deviations for bands
    
    Returns:
        Tuple of (middle_band, upper_band, lower_band)
    """
    middle_band = pd.Series(prices).rolling(window=window).mean()
    std = pd.Series(prices).rolling(window=window).std()
    
    upper_band = middle_band + (num_std * std)
    lower_band = middle_band - (num_std * std)
    
    return middle_band, upper_band, lower_band

def generate_bollinger_signals(prices, middle, upper, lower):
    """
    Generate signals from Bollinger Bands.
    
    Args:
        prices: Price series
        middle: Middle band
        upper: Upper band
        lower: Lower band
    
    Returns:
        Signal series: +1 (long), -1 (short), 0 (neutral)
    """
    signals = np.zeros(len(prices))
    
    # Buy when price <= lower band
    signals[prices <= lower] = 1
    
    # Sell when price >= upper band
    signals[prices >= upper] = -1
    
    return signals

# Use the same price series from previous section
middle, upper, lower = calculate_bollinger_bands(prices, window=20, num_std=2)
bb_signals = generate_bollinger_signals(prices, middle, upper, lower)

# Calculate % Bollinger (%B): shows where price is within the bands
# %B = (price - lower) / (upper - lower)
# %B > 1: above upper band
# %B < 0: below lower band
# %B = 0.5: at middle band
pct_b = (prices - lower) / (upper - lower)

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# 1. Price with Bollinger Bands
axes[0].plot(prices, linewidth=1.5, color='black', label='Price', zorder=3)
axes[0].plot(middle, linewidth=1.5, color='blue', linestyle='--', label='Middle (MA20)', alpha=0.7)
axes[0].plot(upper, linewidth=1, color='red', linestyle='--', label='Upper (+2σ)', alpha=0.7)
axes[0].plot(lower, linewidth=1, color='green', linestyle='--', label='Lower (-2σ)', alpha=0.7)
axes[0].fill_between(range(len(prices)), lower, upper, alpha=0.1, color='gray')

# Mark buy/sell signals
buy_points = np.where(bb_signals > 0)[0]
sell_points = np.where(bb_signals < 0)[0]
axes[0].scatter(buy_points, prices[buy_points], color='green', marker='^', s=50, alpha=0.5, label='Buy', zorder=4)
axes[0].scatter(sell_points, prices[sell_points], color='red', marker='v', s=50, alpha=0.5, label='Sell', zorder=4)

axes[0].set_title('Bollinger Bands (20-day, 2σ)', fontweight='bold')
axes[0].set_ylabel('Price')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# 2. %B Indicator
axes[1].plot(pct_b, linewidth=1.5, color='purple', label='%B')
axes[1].axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Upper band')
axes[1].axhline(0.0, color='green', linestyle='--', alpha=0.5, label='Lower band')
axes[1].axhline(0.5, color='blue', linestyle='--', alpha=0.5, label='Middle band')
axes[1].fill_between(range(len(pct_b)), 0, 1, alpha=0.1, color='gray')
axes[1].set_title('%B Indicator (position within bands)', fontweight='bold')
axes[1].set_ylabel('%B')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Band Width (volatility measure)
band_width = (upper - lower) / middle
axes[2].plot(band_width, linewidth=1.5, color='orange', label='Band Width')
axes[2].set_title('Band Width (volatility measure)', fontweight='bold')
axes[2].set_ylabel('Width / MA')
axes[2].set_xlabel('Time (periods)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Bollinger Band Statistics:")
print("=" * 60)
print(f"Times price touched upper band: {np.sum(prices >= upper):.0f}")
print(f"Times price touched lower band: {np.sum(prices <= lower):.0f}")
print(f"Times price was outside bands: {np.sum((prices >= upper) | (prices <= lower)):.0f}")
print(f"Percentage of time outside bands: {np.sum((prices >= upper) | (prices <= lower)) / len(prices) * 100:.1f}%")
print(f"\nAverage band width: {np.nanmean(band_width):.3f}")
print(f"Max band width (high volatility): {np.nanmax(band_width):.3f}")
print(f"Min band width (low volatility): {np.nanmin(band_width):.3f}")

print("\n💡 Key Insight: Bands widen during volatile periods and narrow during calm periods")
print("   This auto-adjusts entry thresholds based on market conditions!")

---

## 4. Entry Threshold Optimization

Now let's **systematically test** different entry thresholds to see which performs best.

We'll simulate forward returns and calculate:
- **Information Coefficient (IC)**: Correlation between signal and future returns
- **Sharpe Ratio**: Risk-adjusted returns
- **Win Rate**: Percentage of profitable trades
- **Turnover**: How often we trade

In [ ]:
def simulate_mean_reverting_returns(prices, half_life=10):
    """
    Simulate forward returns with mean-reverting property.
    
    Returns are inversely correlated with current Z-score:
    - High Z (overbought) → negative future returns
    - Low Z (oversold) → positive future returns
    """
    # Calculate Z-scores
    window = 20
    zscore = calculate_zscore(prices, window=window)
    
    # Forward returns are inversely proportional to Z-score
    # Add noise to make it realistic
    mean_reversion_strength = 0.1  # IC ≈ 0.05-0.10 for mean reversion
    noise_level = 0.02  # Market noise
    
    returns = -mean_reversion_strength * zscore + np.random.normal(0, noise_level, len(prices))
    
    return returns

def backtest_threshold(prices, returns, threshold, holding_period=5):
    """
    Backtest a Z-score threshold strategy.
    
    Args:
        prices: Price series
        returns: Forward returns
        threshold: Z-score entry threshold
        holding_period: How long to hold position
    
    Returns:
        Dict with performance metrics
    """
    zscore = calculate_zscore(prices, window=20)
    signals = generate_mean_reversion_signals(zscore, threshold=threshold)
    
    # Calculate strategy returns
    # Signal at t, return from t to t+holding_period
    strategy_returns = []
    signal_values = []
    
    for i in range(len(signals) - holding_period):
        if signals[i] != 0:
            # Cumulative return over holding period
            cum_return = np.sum(returns[i:i+holding_period])
            strategy_returns.append(signals[i] * cum_return)
            signal_values.append(signals[i])
    
    if len(strategy_returns) == 0:
        return {
            'threshold': threshold,
            'ic': 0.0,
            'sharpe': 0.0,
            'win_rate': 0.0,
            'avg_return': 0.0,
            'num_trades': 0,
            'turnover': 0.0
        }
    
    # Calculate metrics
    ic = calculate_ic(signal_values, strategy_returns)
    sharpe = np.mean(strategy_returns) / np.std(strategy_returns) * np.sqrt(252 / holding_period) if np.std(strategy_returns) > 0 else 0
    win_rate = np.sum(np.array(strategy_returns) > 0) / len(strategy_returns)
    avg_return = np.mean(strategy_returns)
    num_trades = len(strategy_returns)
    turnover = np.sum(np.abs(np.diff(signals))) / len(signals)  # Average daily turnover
    
    return {
        'threshold': threshold,
        'ic': ic,
        'sharpe': sharpe,
        'win_rate': win_rate,
        'avg_return': avg_return,
        'num_trades': num_trades,
        'turnover': turnover
    }

# Generate mean-reverting returns
returns = simulate_mean_reverting_returns(prices)

# Test different thresholds
thresholds_to_test = np.arange(1.0, 3.5, 0.25)
results = []

for threshold in thresholds_to_test:
    result = backtest_threshold(prices, returns, threshold, holding_period=5)
    results.append(result)

results_df = pd.DataFrame(results)

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. IC
axes[0, 0].plot(results_df['threshold'], results_df['ic'], marker='o', linewidth=2, color='steelblue')
axes[0, 0].axhline(0, color='red', linestyle='--', alpha=0.3)
axes[0, 0].set_title('Information Coefficient vs Threshold', fontweight='bold')
axes[0, 0].set_xlabel('Z-score Threshold')
axes[0, 0].set_ylabel('IC')
axes[0, 0].grid(True, alpha=0.3)

# 2. Sharpe Ratio
axes[0, 1].plot(results_df['threshold'], results_df['sharpe'], marker='o', linewidth=2, color='green')
axes[0, 1].axhline(0, color='red', linestyle='--', alpha=0.3)
axes[0, 1].axhline(1, color='orange', linestyle='--', alpha=0.3, label='Sharpe = 1')
axes[0, 1].set_title('Sharpe Ratio vs Threshold', fontweight='bold')
axes[0, 1].set_xlabel('Z-score Threshold')
axes[0, 1].set_ylabel('Sharpe Ratio')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Win Rate
axes[0, 2].plot(results_df['threshold'], results_df['win_rate'], marker='o', linewidth=2, color='purple')
axes[0, 2].axhline(0.5, color='orange', linestyle='--', alpha=0.3, label='50% (random)')
axes[0, 2].set_title('Win Rate vs Threshold', fontweight='bold')
axes[0, 2].set_xlabel('Z-score Threshold')
axes[0, 2].set_ylabel('Win Rate')
axes[0, 2].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# 4. Number of Trades
axes[1, 0].plot(results_df['threshold'], results_df['num_trades'], marker='o', linewidth=2, color='orange')
axes[1, 0].set_title('Number of Trades vs Threshold', fontweight='bold')
axes[1, 0].set_xlabel('Z-score Threshold')
axes[1, 0].set_ylabel('Number of Trades')
axes[1, 0].grid(True, alpha=0.3)

# 5. Turnover
axes[1, 1].plot(results_df['threshold'], results_df['turnover'], marker='o', linewidth=2, color='red')
axes[1, 1].set_title('Turnover vs Threshold', fontweight='bold')
axes[1, 1].set_xlabel('Z-score Threshold')
axes[1, 1].set_ylabel('Daily Turnover')
axes[1, 1].grid(True, alpha=0.3)

# 6. Average Return per Trade
axes[1, 2].plot(results_df['threshold'], results_df['avg_return'], marker='o', linewidth=2, color='teal')
axes[1, 2].axhline(0, color='red', linestyle='--', alpha=0.3)
axes[1, 2].set_title('Average Return per Trade vs Threshold', fontweight='bold')
axes[1, 2].set_xlabel('Z-score Threshold')
axes[1, 2].set_ylabel('Avg Return per Trade')
axes[1, 2].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2%}'))
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find optimal threshold
best_sharpe_idx = results_df['sharpe'].idxmax()
best_ic_idx = results_df['ic'].idxmax()

print("\n📊 Threshold Optimization Results:")
print("=" * 60)
print(f"\nBest Sharpe Ratio: {results_df.loc[best_sharpe_idx, 'sharpe']:.3f} at threshold {results_df.loc[best_sharpe_idx, 'threshold']:.2f}")
print(f"Best IC: {results_df.loc[best_ic_idx, 'ic']:.3f} at threshold {results_df.loc[best_ic_idx, 'threshold']:.2f}")

print("\n" + "=" * 60)
print("Threshold Comparison:")
print("=" * 60)
comparison_thresholds = [1.5, 2.0, 2.5]
for t in comparison_thresholds:
    row = results_df[results_df['threshold'] == t].iloc[0]
    print(f"\nThreshold {t}σ:")
    print(f"  IC: {row['ic']:.4f}")
    print(f"  Sharpe: {row['sharpe']:.3f}")
    print(f"  Win Rate: {row['win_rate']:.1%}")
    print(f"  Trades: {row['num_trades']:.0f}")
    print(f"  Turnover: {row['turnover']:.3f}")

print("\n💡 Key Insight: Higher thresholds typically have:")
print("   ✓ Better signal quality (higher IC)")
print("   ✓ Higher win rate")
print("   ✓ Lower turnover (fewer transactions costs)")
print("   ✗ Fewer trading opportunities")

---

## 5. Sector-Neutral Positioning

**Problem**: Mean reversion signals might be correlated across sectors, leading to concentrated risk.

**Solution**: Force portfolio to be **sector-neutral** (net zero exposure within each sector).

**Benefits**:
- Eliminates systematic risk
- Pure alpha capture (no beta exposure)
- More stable performance across market regimes

**Implementation**:
1. Calculate Z-scores for all instruments
2. Within each sector, go long the most oversold and short the most overbought
3. Ensure net exposure within sector = 0

In [ ]:
def create_sector_neutral_portfolio(prices_dict, sectors_dict, threshold=2.0):
    """
    Create sector-neutral mean reversion portfolio.
    
    Args:
        prices_dict: Dict of {instrument: price_series}
        sectors_dict: Dict of {instrument: sector}
        threshold: Z-score entry threshold
    
    Returns:
        Dict of {instrument: position} where sum(positions per sector) = 0
    """
    # Calculate Z-scores for all instruments
    zscores = {}
    for instrument, prices in prices_dict.items():
        zscore = calculate_zscore(prices, window=20)
        zscores[instrument] = zscore.iloc[-1] if hasattr(zscore, 'iloc') else zscore[-1]
    
    # Group by sector
    sectors = set(sectors_dict.values())
    positions = {}
    
    for sector in sectors:
        # Get instruments in this sector
        sector_instruments = [inst for inst, sec in sectors_dict.items() if sec == sector]
        
        # Sort by Z-score
        sector_zscores = [(inst, zscores[inst]) for inst in sector_instruments]
        sector_zscores.sort(key=lambda x: x[1])
        
        # Long most oversold (lowest Z), short most overbought (highest Z)
        for inst, z in sector_zscores:
            if z < -threshold:
                positions[inst] = 1.0  # Long (oversold)
            elif z > threshold:
                positions[inst] = -1.0  # Short (overbought)
            else:
                positions[inst] = 0.0
        
        # Ensure sector neutrality
        sector_exposure = sum(positions.get(inst, 0) for inst in sector_instruments)
        if sector_exposure != 0:
            # Adjust positions to make sector neutral
            # Simple approach: scale down to balance
            long_count = sum(1 for inst in sector_instruments if positions.get(inst, 0) > 0)
            short_count = sum(1 for inst in sector_instruments if positions.get(inst, 0) < 0)
            
            if long_count > short_count and short_count > 0:
                # More longs than shorts: scale up shorts
                scale = long_count / short_count
                for inst in sector_instruments:
                    if positions.get(inst, 0) < 0:
                        positions[inst] *= scale
            elif short_count > long_count and long_count > 0:
                # More shorts than longs: scale up longs
                scale = short_count / long_count
                for inst in sector_instruments:
                    if positions.get(inst, 0) > 0:
                        positions[inst] *= scale
    
    return positions

# Simulate multi-sector portfolio
np.random.seed(42)
n_instruments = 12
n_periods = 250

# Create 3 sectors with 4 instruments each
sectors_dict = {}
for i in range(n_instruments):
    sector = ['Energy', 'Financials', 'Technology'][i // 4]
    sectors_dict[f'INST{i+1}'] = sector

# Generate mean-reverting prices for each instrument
prices_dict = {}
for i in range(n_instruments):
    theta = np.random.uniform(0.05, 0.15)
    mu = 100 + np.random.uniform(-5, 5)
    sigma = np.random.uniform(1.5, 3.0)
    
    X = simulate_ou_process(theta, mu, sigma, T=n_periods, dt=1, X0=mu + np.random.uniform(-10, 10))
    prices_dict[f'INST{i+1}'] = X

# Create sector-neutral portfolio
positions = create_sector_neutral_portfolio(prices_dict, sectors_dict, threshold=2.0)

# Verify sector neutrality
sector_exposures = {}
for sector in set(sectors_dict.values()):
    sector_instruments = [inst for inst, sec in sectors_dict.items() if sec == sector]
    exposure = sum(positions.get(inst, 0) for inst in sector_instruments)
    sector_exposures[sector] = exposure

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 1. Positions by instrument, colored by sector
instruments = list(positions.keys())
position_values = [positions[inst] for inst in instruments]
colors = ['red' if sectors_dict[inst] == 'Energy' else 'blue' if sectors_dict[inst] == 'Financials' else 'green' 
          for inst in instruments]

bars = axes[0].bar(range(len(instruments)), position_values, color=colors, alpha=0.7, edgecolor='black')
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('Sector-Neutral Portfolio Positions', fontweight='bold', fontsize=14)
axes[0].set_ylabel('Position Size')
axes[0].set_xticks(range(len(instruments)))
axes[0].set_xticklabels(instruments, rotation=45, ha='right')
axes[0].grid(True, alpha=0.3, axis='y')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', alpha=0.7, label='Energy'),
    Patch(facecolor='blue', alpha=0.7, label='Financials'),
    Patch(facecolor='green', alpha=0.7, label='Technology')
]
axes[0].legend(handles=legend_elements, loc='upper left')

# 2. Sector exposures (should all be near zero)
sectors = list(sector_exposures.keys())
exposures = [sector_exposures[s] for s in sectors]
colors_sector = ['red', 'blue', 'green']

axes[1].bar(sectors, exposures, color=colors_sector, alpha=0.7, edgecolor='black')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('Net Sector Exposures (should be ~0)', fontweight='bold', fontsize=14)
axes[1].set_ylabel('Net Exposure')
axes[1].set_ylim(-0.5, 0.5)
axes[1].grid(True, alpha=0.3, axis='y')

# Add text showing exact exposures
for i, (sector, exposure) in enumerate(sector_exposures.items()):
    axes[1].text(i, exposure + 0.05 if exposure >= 0 else exposure - 0.05, 
                f'{exposure:.2f}', ha='center', va='bottom' if exposure >= 0 else 'top',
                fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Sector-Neutral Portfolio Analysis:")
print("=" * 60)
for sector in sorted(sector_exposures.keys()):
    sector_instruments = [inst for inst, sec in sectors_dict.items() if sec == sector]
    long_count = sum(1 for inst in sector_instruments if positions.get(inst, 0) > 0)
    short_count = sum(1 for inst in sector_instruments if positions.get(inst, 0) < 0)
    neutral_count = sum(1 for inst in sector_instruments if positions.get(inst, 0) == 0)
    
    print(f"\n{sector}:")
    print(f"  Long: {long_count}, Short: {short_count}, Neutral: {neutral_count}")
    print(f"  Net exposure: {sector_exposures[sector]:.3f}")

total_long = sum(1 for p in position_values if p > 0)
total_short = sum(1 for p in position_values if p < 0)

print(f"\nTotal portfolio:")
print(f"  Long positions: {total_long}")
print(f"  Short positions: {total_short}")
print(f"  Gross exposure: {sum(abs(p) for p in position_values):.2f}")
print(f"  Net exposure: {sum(position_values):.3f}")

print("\n💡 Key Insight: Sector-neutral portfolios eliminate market beta")
print("   Returns come purely from stock selection (alpha), not sector moves (beta)")

---

## 6. High Turnover Management & Transaction Costs

Mean reversion strategies have **high turnover** because:
- Positions reverse frequently (long → short → long)
- Entry/exit triggers fire often
- Short holding periods

**Problem**: Transaction costs can destroy profits!

**Solutions**:
1. **Widen entry thresholds** (2.5σ instead of 2σ)
2. **Add position hold time** (minimum 3-5 days)
3. **Use limit orders** (don't cross the spread)
4. **Portfolio rebalancing bands** (don't rebalance tiny changes)

Let's quantify the impact!

In [ ]:
def calculate_turnover(positions):
    """
    Calculate turnover from position time series.
    
    Turnover = sum(|Δposition|) / 2
    (Divide by 2 because buying and selling are counted separately)
    """
    if len(positions) < 2:
        return 0.0
    
    position_changes = np.diff(positions, axis=0)
    turnover = np.sum(np.abs(position_changes), axis=1) / 2
    
    return turnover

def apply_transaction_costs(returns, turnover, cost_bps=5):
    """
    Apply transaction costs to returns.
    
    Args:
        returns: Gross returns
        turnover: Turnover at each period
        cost_bps: Transaction cost in basis points (5 bps = 0.05%)
    
    Returns:
        Net returns after costs
    """
    cost_rate = cost_bps / 10000  # Convert bps to decimal
    costs = turnover * cost_rate
    net_returns = returns - costs
    
    return net_returns

# Simulate strategy with different turnover levels
n_periods = 252

# Generate price series
theta = 0.1
mu = 100
sigma = 2
prices_turnover = simulate_ou_process(theta, mu, sigma, T=n_periods, dt=1, X0=105)
zscore_turnover = calculate_zscore(prices_turnover, window=20)
returns_turnover = simulate_mean_reverting_returns(prices_turnover)

# Strategy 1: Aggressive (low threshold, high turnover)
positions_aggressive = generate_mean_reversion_signals(zscore_turnover, threshold=1.5)
turnover_aggressive = calculate_turnover(positions_aggressive.reshape(-1, 1))

# Strategy 2: Moderate (medium threshold)
positions_moderate = generate_mean_reversion_signals(zscore_turnover, threshold=2.0)
turnover_moderate = calculate_turnover(positions_moderate.reshape(-1, 1))

# Strategy 3: Conservative (high threshold, low turnover)
positions_conservative = generate_mean_reversion_signals(zscore_turnover, threshold=2.5)
turnover_conservative = calculate_turnover(positions_conservative.reshape(-1, 1))

# Calculate strategy returns (simplified: assume position earns next period's return)
strategy_returns_aggressive = positions_aggressive[:-1] * returns_turnover[1:]
strategy_returns_moderate = positions_moderate[:-1] * returns_turnover[1:]
strategy_returns_conservative = positions_conservative[:-1] * returns_turnover[1:]

# Apply transaction costs at different levels
cost_levels = [0, 2, 5, 10]  # basis points
results_costs = []

for cost in cost_levels:
    # Ensure turnover arrays match return arrays
    net_ret_agg = apply_transaction_costs(strategy_returns_aggressive, turnover_aggressive[:-1], cost)
    net_ret_mod = apply_transaction_costs(strategy_returns_moderate, turnover_moderate[:-1], cost)
    net_ret_con = apply_transaction_costs(strategy_returns_conservative, turnover_conservative[:-1], cost)
    
    results_costs.append({
        'cost_bps': cost,
        'aggressive_sharpe': np.mean(net_ret_agg) / np.std(net_ret_agg) * np.sqrt(252) if np.std(net_ret_agg) > 0 else 0,
        'moderate_sharpe': np.mean(net_ret_mod) / np.std(net_ret_mod) * np.sqrt(252) if np.std(net_ret_mod) > 0 else 0,
        'conservative_sharpe': np.mean(net_ret_con) / np.std(net_ret_con) * np.sqrt(252) if np.std(net_ret_con) > 0 else 0,
        'aggressive_return': np.sum(net_ret_agg),
        'moderate_return': np.sum(net_ret_mod),
        'conservative_return': np.sum(net_ret_con),
    })

results_costs_df = pd.DataFrame(results_costs)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Turnover comparison
turnover_comparison = [
    np.mean(turnover_aggressive),
    np.mean(turnover_moderate),
    np.mean(turnover_conservative)
]
strategy_names = ['Aggressive\n(1.5σ)', 'Moderate\n(2.0σ)', 'Conservative\n(2.5σ)']
colors_turn = ['red', 'orange', 'green']

axes[0, 0].bar(strategy_names, turnover_comparison, color=colors_turn, alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Average Daily Turnover by Strategy', fontweight='bold')
axes[0, 0].set_ylabel('Turnover')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 2. Sharpe ratio vs transaction costs
axes[0, 1].plot(results_costs_df['cost_bps'], results_costs_df['aggressive_sharpe'], 
                marker='o', linewidth=2, label='Aggressive', color='red')
axes[0, 1].plot(results_costs_df['cost_bps'], results_costs_df['moderate_sharpe'], 
                marker='s', linewidth=2, label='Moderate', color='orange')
axes[0, 1].plot(results_costs_df['cost_bps'], results_costs_df['conservative_sharpe'], 
                marker='^', linewidth=2, label='Conservative', color='green')
axes[0, 1].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[0, 1].set_title('Sharpe Ratio vs Transaction Costs', fontweight='bold')
axes[0, 1].set_xlabel('Transaction Cost (bps)')
axes[0, 1].set_ylabel('Sharpe Ratio')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Total returns vs transaction costs
axes[1, 0].plot(results_costs_df['cost_bps'], results_costs_df['aggressive_return'], 
                marker='o', linewidth=2, label='Aggressive', color='red')
axes[1, 0].plot(results_costs_df['cost_bps'], results_costs_df['moderate_return'], 
                marker='s', linewidth=2, label='Moderate', color='orange')
axes[1, 0].plot(results_costs_df['cost_bps'], results_costs_df['conservative_return'], 
                marker='^', linewidth=2, label='Conservative', color='green')
axes[1, 0].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[1, 0].set_title('Total Return vs Transaction Costs', fontweight='bold')
axes[1, 0].set_xlabel('Transaction Cost (bps)')
axes[1, 0].set_ylabel('Total Return')
axes[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Position changes over time (aggressive vs conservative)
axes[1, 1].plot(positions_aggressive[:100], linewidth=1.5, label='Aggressive (1.5σ)', 
                color='red', alpha=0.7, drawstyle='steps-post')
axes[1, 1].plot(positions_conservative[:100], linewidth=1.5, label='Conservative (2.5σ)', 
                color='green', alpha=0.7, drawstyle='steps-post')
axes[1, 1].axhline(0, color='black', linestyle='-', alpha=0.3)
axes[1, 1].set_title('Position Changes Over Time (First 100 Days)', fontweight='bold')
axes[1, 1].set_xlabel('Time (days)')
axes[1, 1].set_ylabel('Position')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Turnover Analysis:")
print("=" * 60)
print(f"Aggressive (1.5σ): Avg turnover = {np.mean(turnover_aggressive):.3f} per day")
print(f"Moderate (2.0σ):   Avg turnover = {np.mean(turnover_moderate):.3f} per day")
print(f"Conservative (2.5σ): Avg turnover = {np.mean(turnover_conservative):.3f} per day")

print("\n📊 Impact of Transaction Costs:")
print("=" * 60)
print(results_costs_df.to_string(index=False))

print("\n💡 Key Insight: High turnover strategies are very sensitive to transaction costs!")
print("   At 10 bps, aggressive strategy may become unprofitable")
print("   Conservative strategies are more robust to costs")

---

## 7. Stop-Loss Implementation

**Problem**: Mean reversion assumes prices will return to mean. But what if they don't?

**Danger**: Trending breakouts can cause catastrophic losses!

**Solution**: Implement **stop-losses** to exit when:
- Z-score moves FURTHER away (trend is strengthening)
- Loss exceeds threshold (e.g., -2%)
- Time limit reached (position held too long without reversion)

This is the **most important risk control** for mean reversion!

In [ ]:
def backtest_with_stop_loss(prices, returns, threshold=2.0, stop_loss_z=3.5, stop_loss_pct=0.03, max_hold_days=10):
    """
    Backtest mean reversion with stop-loss rules.
    
    Stop-loss triggers:
    1. Z-score exceeds stop_loss_z (trend is too strong)
    2. Loss exceeds stop_loss_pct (cut losses)
    3. Position held longer than max_hold_days (give up)
    
    Args:
        prices: Price series
        returns: Return series
        threshold: Entry threshold
        stop_loss_z: Z-score stop-loss level
        stop_loss_pct: Percentage loss stop-loss
        max_hold_days: Maximum holding period
    
    Returns:
        Strategy returns, positions, stop-loss triggers
    """
    zscore = calculate_zscore(prices, window=20)
    
    positions = np.zeros(len(prices))
    strategy_returns = np.zeros(len(returns))
    stop_loss_triggers = np.zeros(len(prices))
    
    current_position = 0
    entry_price = 0
    hold_days = 0
    
    for i in range(20, len(prices) - 1):  # Start after zscore warmup
        z = zscore.iloc[i] if hasattr(zscore, 'iloc') else zscore[i]
        
        # If we have a position, check stop-loss conditions
        if current_position != 0:
            hold_days += 1
            
            # Calculate current P&L
            pnl_pct = (prices[i] - entry_price) / entry_price * current_position
            
            # Stop-loss condition 1: Z-score moved further away (trend strengthening)
            if abs(z) > stop_loss_z:
                current_position = 0
                stop_loss_triggers[i] = 1
                hold_days = 0
            
            # Stop-loss condition 2: Loss exceeds threshold
            elif pnl_pct < -stop_loss_pct:
                current_position = 0
                stop_loss_triggers[i] = 2
                hold_days = 0
            
            # Stop-loss condition 3: Max holding period
            elif hold_days >= max_hold_days:
                current_position = 0
                stop_loss_triggers[i] = 3
                hold_days = 0
            
            # Normal exit: Z-score crossed zero (mean reversion completed)
            elif (current_position > 0 and z > 0) or (current_position < 0 and z < 0):
                current_position = 0
                hold_days = 0
        
        # If no position, check entry conditions
        else:
            if z < -threshold:  # Oversold → buy
                current_position = 1
                entry_price = prices[i]
                hold_days = 0
            elif z > threshold:  # Overbought → sell
                current_position = -1
                entry_price = prices[i]
                hold_days = 0
        
        positions[i] = current_position
        
        # Calculate strategy return
        if current_position != 0:
            strategy_returns[i+1] = current_position * returns[i+1]
    
    return strategy_returns, positions, stop_loss_triggers

# Simulate trending breakout scenario
np.random.seed(42)
n = 300
prices_breakout = np.zeros(n)
prices_breakout[0] = 100

# First 100 days: mean-reverting
for i in range(1, 100):
    dX = 0.1 * (100 - prices_breakout[i-1]) + np.random.normal(0, 2)
    prices_breakout[i] = prices_breakout[i-1] + dX

# Days 100-150: Strong upward trend (mean reversion fails!)
for i in range(100, 150):
    dX = 0.5 + np.random.normal(0, 2)  # Drift up
    prices_breakout[i] = prices_breakout[i-1] + dX

# Days 150-300: Back to mean-reverting
for i in range(150, n):
    dX = 0.1 * (120 - prices_breakout[i-1]) + np.random.normal(0, 2)
    prices_breakout[i] = prices_breakout[i-1] + dX

# Calculate returns
returns_breakout = np.diff(prices_breakout) / prices_breakout[:-1]
returns_breakout = np.concatenate([[0], returns_breakout])  # Pad to match length

# Backtest WITHOUT stop-loss
zscore_breakout = calculate_zscore(prices_breakout, window=20)
positions_no_stop = generate_mean_reversion_signals(zscore_breakout, threshold=2.0)
returns_no_stop = positions_no_stop[:-1] * returns_breakout[1:]

# Backtest WITH stop-loss
returns_with_stop, positions_with_stop, stop_triggers = backtest_with_stop_loss(
    prices_breakout, returns_breakout, threshold=2.0, stop_loss_z=3.5, stop_loss_pct=0.03, max_hold_days=10
)

# Calculate cumulative returns
cum_returns_no_stop = (1 + returns_no_stop).cumprod()
cum_returns_with_stop = (1 + returns_with_stop).cumprod()

# Visualization
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

# 1. Price and regime
axes[0].plot(prices_breakout, linewidth=1.5, color='black')
axes[0].axvspan(100, 150, alpha=0.3, color='red', label='Trending Regime (breakout)')
axes[0].set_title('Price Series with Trending Breakout (Days 100-150)', fontweight='bold')
axes[0].set_ylabel('Price')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Z-score with stop-loss level
axes[1].plot(zscore_breakout, linewidth=1.5, color='steelblue')
axes[1].axhline(0, color='black', linestyle='-', alpha=0.3)
axes[1].axhline(2, color='green', linestyle='--', alpha=0.5, label='Entry threshold (±2σ)')
axes[1].axhline(-2, color='green', linestyle='--', alpha=0.5)
axes[1].axhline(3.5, color='red', linestyle='--', alpha=0.5, label='Stop-loss (±3.5σ)')
axes[1].axhline(-3.5, color='red', linestyle='--', alpha=0.5)
axes[1].axvspan(100, 150, alpha=0.2, color='red')
axes[1].set_title('Z-Score (shows extreme deviation during breakout)', fontweight='bold')
axes[1].set_ylabel('Z-score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Positions comparison
axes[2].plot(positions_no_stop, linewidth=1.5, label='Without stop-loss', color='orange', alpha=0.7, drawstyle='steps-post')
axes[2].plot(positions_with_stop, linewidth=1.5, label='With stop-loss', color='green', alpha=0.7, drawstyle='steps-post')
axes[2].scatter(np.where(stop_triggers > 0)[0], np.zeros(np.sum(stop_triggers > 0)), 
                color='red', marker='x', s=100, label='Stop-loss triggered', zorder=5)
axes[2].axhline(0, color='black', linestyle='-', alpha=0.3)
axes[2].axvspan(100, 150, alpha=0.2, color='red')
axes[2].set_title('Position Comparison (Stop-loss exits during trend)', fontweight='bold')
axes[2].set_ylabel('Position')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# 4. Cumulative returns
axes[3].plot(cum_returns_no_stop, linewidth=2, label='Without stop-loss', color='orange')
axes[3].plot(cum_returns_with_stop, linewidth=2, label='With stop-loss', color='green')
axes[3].axhline(1, color='black', linestyle='--', alpha=0.3)
axes[3].axvspan(100, 150, alpha=0.2, color='red')
axes[3].set_title('Cumulative Returns (Stop-loss protects during breakout)', fontweight='bold')
axes[3].set_ylabel('Cumulative Return')
axes[3].set_xlabel('Time (days)')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate performance metrics
sharpe_no_stop = np.mean(returns_no_stop) / np.std(returns_no_stop) * np.sqrt(252) if np.std(returns_no_stop) > 0 else 0
sharpe_with_stop = np.mean(returns_with_stop) / np.std(returns_with_stop) * np.sqrt(252) if np.std(returns_with_stop) > 0 else 0

total_return_no_stop = cum_returns_no_stop[-1] - 1
total_return_with_stop = cum_returns_with_stop[-1] - 1

max_dd_no_stop = np.min(cum_returns_no_stop / np.maximum.accumulate(cum_returns_no_stop) - 1)
max_dd_with_stop = np.min(cum_returns_with_stop / np.maximum.accumulate(cum_returns_with_stop) - 1)

print("\n📊 Stop-Loss Performance Comparison:")
print("=" * 60)
print(f"{'Metric':<30} {'Without Stop':<15} {'With Stop':<15}")
print("=" * 60)
print(f"{'Sharpe Ratio':<30} {sharpe_no_stop:>14.3f} {sharpe_with_stop:>14.3f}")
print(f"{'Total Return':<30} {total_return_no_stop:>13.2%} {total_return_with_stop:>13.2%}")
print(f"{'Max Drawdown':<30} {max_dd_no_stop:>13.2%} {max_dd_with_stop:>13.2%}")

# Count stop-loss triggers by type
n_z_stops = np.sum(stop_triggers == 1)
n_loss_stops = np.sum(stop_triggers == 2)
n_time_stops = np.sum(stop_triggers == 3)

print(f"\n📊 Stop-Loss Triggers:")
print("=" * 60)
print(f"Z-score exceeded {stop_loss_z}σ: {n_z_stops} times")
print(f"Loss exceeded {stop_loss_pct:.0%}: {n_loss_stops} times")
print(f"Max holding period reached: {n_time_stops} times")
print(f"Total stop-loss exits: {n_z_stops + n_loss_stops + n_time_stops}")

print("\n💡 Key Insight: Stop-losses are ESSENTIAL for mean reversion!")
print("   Without stops, trending breakouts cause catastrophic losses")
print("   With stops, we limit losses and preserve capital for true reversions")

---

## 8. When Mean Reversion Works vs. Fails: Regime Detection

**Critical Question**: How do we know if we're in a mean-reverting regime or trending regime?

**Regime Indicators**:
1. **Half-Life**: Short half-life (<10 days) → mean-reverting
2. **Autocorrelation**: Negative autocorrelation → mean-reverting
3. **Hurst Exponent**: H < 0.5 → mean-reverting, H > 0.5 → trending
4. **Variance Ratio**: VR < 1 → mean-reverting, VR > 1 → trending

Let's implement regime detection!

In [ ]:
def calculate_hurst_exponent(prices, lags=range(2, 100)):
    """
    Calculate Hurst exponent using rescaled range analysis.
    
    H < 0.5: Mean-reverting (anti-persistent)
    H = 0.5: Random walk
    H > 0.5: Trending (persistent)
    """
    # Convert to log returns
    tau = []
    lagvec = []
    
    # Calculate the array of the variances of the lagged differences
    for lag in lags:
        # Get log price differences
        pp = np.subtract(prices[lag:], prices[:-lag])
        lagvec.append(lag)
        tau.append(np.sqrt(np.std(pp)))
    
    # Linear fit to log-log plot
    m = np.polyfit(np.log(lagvec), np.log(tau), 1)
    hurst = m[0]
    
    return hurst

def calculate_variance_ratio(returns, lag=5):
    """
    Calculate variance ratio test.
    
    VR < 1: Mean-reverting
    VR = 1: Random walk
    VR > 1: Trending (momentum)
    """
    # Variance of 1-period returns
    var_1 = np.var(returns, ddof=1)
    
    # Variance of k-period returns
    k_period_returns = []
    for i in range(len(returns) - lag + 1):
        k_period_returns.append(np.sum(returns[i:i+lag]))
    
    var_k = np.var(k_period_returns, ddof=1)
    
    # Variance ratio
    vr = var_k / (lag * var_1)
    
    return vr

def detect_regime(prices, window=60):
    """
    Detect market regime using multiple indicators.
    
    Returns:
        'mean_reverting', 'random_walk', or 'trending'
    """
    # 1. Half-life
    hl = estimate_half_life(prices[-window:])
    
    # 2. Hurst exponent
    hurst = calculate_hurst_exponent(prices[-window:])
    
    # 3. Variance ratio
    returns = np.diff(prices[-window:]) / prices[-window:-1]
    vr = calculate_variance_ratio(returns)
    
    # 4. Autocorrelation at lag 1
    autocorr = np.corrcoef(prices[-window:-1], prices[-window+1:])[0, 1]
    
    # Scoring system
    score = 0
    
    # Half-life
    if hl < 10:
        score -= 2  # Mean-reverting
    elif hl > 30:
        score += 2  # Trending
    
    # Hurst
    if hurst < 0.45:
        score -= 2  # Mean-reverting
    elif hurst > 0.55:
        score += 2  # Trending
    
    # Variance ratio
    if vr < 0.9:
        score -= 1  # Mean-reverting
    elif vr > 1.1:
        score += 1  # Trending
    
    # Autocorrelation
    if autocorr < 0.9:
        score -= 1  # Mean-reverting
    elif autocorr > 0.98:
        score += 1  # Trending
    
    # Determine regime
    if score <= -3:
        regime = 'mean_reverting'
    elif score >= 3:
        regime = 'trending'
    else:
        regime = 'random_walk'
    
    return regime, {'half_life': hl, 'hurst': hurst, 'variance_ratio': vr, 'autocorr': autocorr, 'score': score}

# Simulate three different regimes
n = 200

# Regime 1: Mean-reverting
prices_mr = simulate_ou_process(theta=0.15, mu=100, sigma=2, T=n, dt=1, X0=110)

# Regime 2: Random walk
prices_rw = np.zeros(n)
prices_rw[0] = 100
for i in range(1, n):
    prices_rw[i] = prices_rw[i-1] + np.random.normal(0, 2)

# Regime 3: Trending
prices_trend = np.zeros(n)
prices_trend[0] = 100
for i in range(1, n):
    prices_trend[i] = prices_trend[i-1] + 0.3 + np.random.normal(0, 2)  # Upward drift

# Detect regimes
regime_mr, metrics_mr = detect_regime(prices_mr)
regime_rw, metrics_rw = detect_regime(prices_rw)
regime_trend, metrics_trend = detect_regime(prices_trend)

# Visualization
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

# Plot each regime
regimes = [
    (prices_mr, 'Mean-Reverting', regime_mr, metrics_mr, 'green'),
    (prices_rw, 'Random Walk', regime_rw, metrics_rw, 'orange'),
    (prices_trend, 'Trending', regime_trend, metrics_trend, 'red')
]

for idx, (prices, title, regime, metrics, color) in enumerate(regimes):
    # Price series
    axes[idx, 0].plot(prices, linewidth=1.5, color=color)
    axes[idx, 0].set_title(f'{title} Regime (Detected: {regime})', fontweight='bold')
    axes[idx, 0].set_ylabel('Price')
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Metrics comparison
    metric_names = ['Half-Life', 'Hurst', 'VR', 'AutoCorr']
    metric_values = [metrics['half_life'], metrics['hurst'], metrics['variance_ratio'], metrics['autocorr']]
    
    # Color-code bars by regime signal
    bar_colors = []
    for name, val in zip(metric_names, metric_values):
        if name == 'Half-Life':
            bar_colors.append('green' if val < 10 else 'red' if val > 30 else 'orange')
        elif name == 'Hurst':
            bar_colors.append('green' if val < 0.45 else 'red' if val > 0.55 else 'orange')
        elif name == 'VR':
            bar_colors.append('green' if val < 0.9 else 'red' if val > 1.1 else 'orange')
        elif name == 'AutoCorr':
            bar_colors.append('green' if val < 0.9 else 'red' if val > 0.98 else 'orange')
    
    axes[idx, 1].bar(metric_names, metric_values, color=bar_colors, alpha=0.7, edgecolor='black')
    axes[idx, 1].set_title(f'Regime Indicators (Score: {metrics["score"]})', fontweight='bold')
    axes[idx, 1].set_ylabel('Value')
    axes[idx, 1].grid(True, alpha=0.3, axis='y')
    
    # Add reference lines
    if idx == 0:  # Only on first plot
        axes[idx, 1].text(0.5, 0.95, 'Green=MeanRev, Red=Trend, Orange=Neutral',
                         transform=axes[idx, 1].transAxes, ha='center', va='top',
                         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[-1, 0].set_xlabel('Time (days)')

plt.tight_layout()
plt.show()

print("\n📊 Regime Detection Results:")
print("=" * 80)
print(f"{'Regime':<20} {'Detected':<15} {'Half-Life':<12} {'Hurst':<10} {'VR':<10} {'AutoCorr':<10} {'Score':<8}")
print("=" * 80)

for prices, title, regime, metrics, color in regimes:
    print(f"{title:<20} {regime:<15} {metrics['half_life']:>10.2f} {metrics['hurst']:>10.3f} "
          f"{metrics['variance_ratio']:>10.3f} {metrics['autocorr']:>10.3f} {metrics['score']:>8.0f}")

print("\n💡 Key Insight: Mean reversion strategies should ONLY trade in mean-reverting regimes!")
print("   Use regime detection to turn strategy on/off dynamically")
print("   In trending regimes, switch to momentum or stay in cash")

---

## 9. Mean Reversion vs Momentum: The Fundamental Comparison

Mean reversion and momentum are **philosophical opposites**:

| Aspect | Mean Reversion | Momentum |
|--------|---------------|----------|
| **Philosophy** | Prices return to average | Trends persist |
| **Signal** | -Z-score (fade extremes) | +Z-score (follow trends) |
| **IC** | 0.02-0.05 (moderate) | 0.05-0.10 (higher) |
| **Holding Period** | Short (1-10 days) | Medium-Long (20-60 days) |
| **Turnover** | High | Low-Medium |
| **Works In** | Ranging markets | Trending markets |
| **Fails In** | Trending breakouts | Choppy/range-bound |
| **Correlation** | **Negative!** | **Negative!** |

**Powerful Insight**: Because they're negatively correlated, combining them can improve portfolio Sharpe ratio!

In [ ]:
# Compare mean reversion vs momentum on the same price series
from Signals.Futures.MomentumSignal import MomentumSignal

# Create mixed regime (ranging → trending → ranging)
n = 400
prices_mixed = np.zeros(n)
prices_mixed[0] = 100

# Phase 1 (0-150): Ranging/mean-reverting
for i in range(1, 150):
    dX = 0.1 * (100 - prices_mixed[i-1]) + np.random.normal(0, 2)
    prices_mixed[i] = prices_mixed[i-1] + dX

# Phase 2 (150-250): Strong uptrend
for i in range(150, 250):
    dX = 0.3 + np.random.normal(0, 2)
    prices_mixed[i] = prices_mixed[i-1] + dX

# Phase 3 (250-400): Ranging again
for i in range(250, n):
    dX = 0.1 * (130 - prices_mixed[i-1]) + np.random.normal(0, 2)
    prices_mixed[i] = prices_mixed[i-1] + dX

# Calculate signals
zscore_mixed = calculate_zscore(prices_mixed, window=20)

# Mean reversion: -Z (fade extremes)
signals_mr = -zscore_mixed  # Invert Z-score

# Momentum: +Z (follow trends)
signals_mom = zscore_mixed  # Use Z-score as-is

# Calculate returns
returns_mixed = np.diff(prices_mixed) / prices_mixed[:-1]
returns_mixed = np.concatenate([[0], returns_mixed])

# Strategy returns (assuming we can scale signals to positions)
strategy_returns_mr = signals_mr[:-1] * returns_mixed[1:] * 0.1  # Scale down for reasonable position sizing
strategy_returns_mom = signals_mom[:-1] * returns_mixed[1:] * 0.1

# Combined strategy (50/50 blend)
strategy_returns_combined = 0.5 * strategy_returns_mr + 0.5 * strategy_returns_mom

# Calculate cumulative returns
cum_mr = (1 + strategy_returns_mr).cumprod()
cum_mom = (1 + strategy_returns_mom).cumprod()
cum_combined = (1 + strategy_returns_combined).cumprod()

# Visualization
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

# 1. Price with regime shading
axes[0].plot(prices_mixed, linewidth=1.5, color='black')
axes[0].axvspan(0, 150, alpha=0.2, color='green', label='Ranging (MR wins)')
axes[0].axvspan(150, 250, alpha=0.2, color='red', label='Trending (Mom wins)')
axes[0].axvspan(250, 400, alpha=0.2, color='green')
axes[0].set_title('Price Series with Regime Changes', fontweight='bold')
axes[0].set_ylabel('Price')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Signals comparison
axes[1].plot(signals_mr, linewidth=1, label='Mean Reversion (-Z)', color='green', alpha=0.7)
axes[1].plot(signals_mom, linewidth=1, label='Momentum (+Z)', color='red', alpha=0.7)
axes[1].axhline(0, color='black', linestyle='-', alpha=0.3)
axes[1].axvspan(0, 150, alpha=0.1, color='green')
axes[1].axvspan(150, 250, alpha=0.1, color='red')
axes[1].axvspan(250, 400, alpha=0.1, color='green')
axes[1].set_title('Signals: Mean Reversion vs Momentum (notice opposite signs)', fontweight='bold')
axes[1].set_ylabel('Signal')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Strategy returns
axes[2].plot(strategy_returns_mr, linewidth=0.5, label='Mean Reversion', color='green', alpha=0.5)
axes[2].plot(strategy_returns_mom, linewidth=0.5, label='Momentum', color='red', alpha=0.5)
axes[2].plot(strategy_returns_combined, linewidth=0.5, label='Combined (50/50)', color='blue', alpha=0.7)
axes[2].axhline(0, color='black', linestyle='-', alpha=0.3)
axes[2].axvspan(0, 150, alpha=0.1, color='green')
axes[2].axvspan(150, 250, alpha=0.1, color='red')
axes[2].axvspan(250, 400, alpha=0.1, color='green')
axes[2].set_title('Daily Returns by Strategy', fontweight='bold')
axes[2].set_ylabel('Return')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# 4. Cumulative returns
axes[3].plot(cum_mr, linewidth=2, label='Mean Reversion', color='green')
axes[3].plot(cum_mom, linewidth=2, label='Momentum', color='red')
axes[3].plot(cum_combined, linewidth=2, label='Combined (50/50)', color='blue', linestyle='--')
axes[3].axhline(1, color='black', linestyle='--', alpha=0.3)
axes[3].axvspan(0, 150, alpha=0.1, color='green')
axes[3].axvspan(150, 250, alpha=0.1, color='red')
axes[3].axvspan(250, 400, alpha=0.1, color='green')
axes[3].set_title('Cumulative Returns (Combined has smoother growth)', fontweight='bold')
axes[3].set_ylabel('Cumulative Return')
axes[3].set_xlabel('Time (days)')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate performance metrics
def calc_metrics(returns, name):
    sharpe = np.mean(returns) / np.std(returns) * np.sqrt(252) if np.std(returns) > 0 else 0
    total_ret = np.sum(returns)
    cum = (1 + returns).cumprod()
    max_dd = np.min(cum / np.maximum.accumulate(cum) - 1)
    
    return {
        'Strategy': name,
        'Sharpe': sharpe,
        'Total Return': total_ret,
        'Max DD': max_dd,
        'Volatility': np.std(returns) * np.sqrt(252)
    }

results_comparison = [
    calc_metrics(strategy_returns_mr, 'Mean Reversion'),
    calc_metrics(strategy_returns_mom, 'Momentum'),
    calc_metrics(strategy_returns_combined, 'Combined (50/50)')
]

df_comparison = pd.DataFrame(results_comparison)

print("\n📊 Performance Comparison: Mean Reversion vs Momentum")
print("=" * 80)
print(df_comparison.to_string(index=False))

# Calculate correlation
corr_mr_mom = np.corrcoef(strategy_returns_mr, strategy_returns_mom)[0, 1]

print(f"\n📊 Strategy Correlation:")
print("=" * 80)
print(f"Correlation(MeanReversion, Momentum): {corr_mr_mom:.3f}")

print("\n💡 KEY INSIGHTS:")
print("   1. Mean reversion and momentum have NEGATIVE correlation")
print("   2. Mean reversion wins in ranging markets, loses in trends")
print("   3. Momentum wins in trends, loses in ranging markets")
print("   4. Combining them creates smoother returns with better Sharpe ratio")
print("   5. Portfolio diversification benefit from negative correlation!")

---

## Summary: Mean Reversion Strategy Playbook

### What We've Learned

1. **Ornstein-Uhlenbeck Process** - Mathematical foundation for mean reversion
   - Half-life formula: τ = -ln(2)/θ
   - Fast reversion (1-5 days) ideal for trading

2. **Z-Score Signals** - Entry/exit based on standard deviations
   - Higher thresholds (2.5σ) = better quality, lower turnover
   - Lower thresholds (1.5σ) = more trades, higher costs

3. **Bollinger Bands** - Volatility-adjusted mean reversion
   - Automatically adapts to market conditions
   - Band width indicates volatility regime

4. **Entry Threshold Optimization** - Systematic testing reveals tradeoffs
   - IC, Sharpe, win rate all improve with higher thresholds
   - But fewer opportunities means lower total returns

5. **Sector-Neutral Portfolios** - Eliminate systematic risk
   - Force net zero exposure within sectors
   - Pure alpha capture without beta

6. **Turnover Management** - Transaction costs can kill profits
   - Mean reversion has inherently high turnover
   - Conservative strategies more robust to costs

7. **Stop-Loss Rules** - ESSENTIAL for survival
   - Z-score stops (exit if |Z| > 3.5)
   - Loss stops (exit if loss > 3%)
   - Time stops (exit after 10 days)
   - Without stops, trending breakouts are catastrophic

8. **Regime Detection** - Know when to turn strategy on/off
   - Half-life, Hurst exponent, variance ratio, autocorrelation
   - Only trade in mean-reverting regimes
   - Switch to momentum or cash in trending regimes

9. **Mean Reversion vs Momentum** - Opposite philosophies, negative correlation
   - Combining them improves Sharpe ratio
   - Diversification benefit from strategy mixing

### When Mean Reversion Works
✅ Ranging markets
✅ High-frequency reversion (short half-life)
✅ High volatility environments (wider bands)
✅ Liquid markets with tight spreads

### When Mean Reversion Fails
❌ Trending breakouts
❌ Structural regime changes
❌ Low volatility traps (false signals)
❌ Illiquid markets (can't exit)

### Best Practices

1. **Always use stop-losses** - Non-negotiable!
2. **Monitor regime** - Turn off in trending markets
3. **Manage turnover** - Use higher thresholds
4. **Combine with momentum** - Negative correlation benefit
5. **Test transaction costs** - They matter more than you think!

---

## Next Steps

- **Implement real-time regime detection** in production
- **Combine mean reversion + momentum** in multi-signal portfolio
- **Test on real futures data** (not just simulations)
- **Optimize stop-loss parameters** for your specific market
- **Monitor IC decay** - mean reversion can stop working suddenly!
